# Cold-Start Ranking in Fraud & Credit Risk
## Notebook 2: IEEE-CIS Fraud Detection Dataset (Real Data)

This notebook replicates the cold-start placement experiment on **590,540 real financial transactions** from the IEEE-CIS Fraud Detection competition (Vesta Corporation).

### Why Real Data Matters
The synthetic simulation proved the *effect exists in principle*. Real data tells us:
- Does it survive messy, imbalanced, real-world fraud rates?
- How fast do estimates converge on real transaction volumes?
- Do the same strategy rankings hold?

### Key Differences from Notebook 1
| | Synthetic | IEEE-CIS |
|---|---|---|
| Data source | Generated | Real Vesta transactions |
| Fraud rate | ~5% (simulated) | 3.5% (real, imbalanced) |
| Score model | Gaussian conjugate | Beta-Binomial conjugate |
| Cold entity definition | Top 30% true score | Cards with 2–5 real transactions |
| Interactions | Continuous observations | Discrete fraud/not-fraud per transaction |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import kendalltau, beta as beta_dist
import warnings
warnings.filterwarnings('ignore')

# Parameters
SEED        = 42
N_WARM      = 150
N_COLD      = 60
T_STEPS     = 20
K           = 20
LCB_K       = 1.5
TIER_BAND   = (120, 140)
GRAD_THRESH = 8
CONV_TOL    = 5
N_RUNS      = 20
PRIOR_A     = 0.5    # Beta prior alpha: encodes ~3.5% population fraud rate
PRIOR_B     = 13.5   # Beta prior beta

STRATEGIES = ['Naive','LCB','Tiered','Random']
COLORS     = {'Naive':'#E74C3C','LCB':'#2ECC71','Tiered':'#3498DB','Random':'#95A5A6'}

# UPDATE THIS PATH to where you saved train_transaction.csv
DATA_PATH = 'train_transaction.csv/train_transaction.csv'
print('Ready.')

## Step 1: Load and Explore the Data

In [ ]:
print('Loading dataset...')
df = pd.read_csv(DATA_PATH, usecols=['isFraud','card1','TransactionAmt'])
print(f'Total transactions: {len(df):,}')
print(f'Overall fraud rate: {df.isFraud.mean():.2%}')
print(f'Total unique cards (card1): {df.card1.nunique():,}')

# Group by card
cs = df.groupby('card1').agg(
    n_tx    = ('isFraud','count'),
    n_fraud = ('isFraud','sum'),
    avg_amt = ('TransactionAmt','mean')
).reset_index()
cs['true_rate'] = cs['n_fraud'] / cs['n_tx']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(np.log1p(cs.n_tx), bins=40, color='#3498DB', alpha=0.7, edgecolor='white')
axes[0].axvline(np.log1p(10), color='red', ls='--', label='Warm threshold (10 tx)')
axes[0].axvline(np.log1p(5),  color='orange', ls='--', label='Cold upper bound (5 tx)')
axes[0].set_title('Transaction Count per Card (log scale)', fontweight='bold')
axes[0].set_xlabel('log(1 + n_transactions)'); axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

axes[1].hist(cs.true_rate, bins=30, color='#E74C3C', alpha=0.7, edgecolor='white')
axes[1].set_title('Per-Card True Fraud Rate Distribution', fontweight='bold')
axes[1].set_xlabel('True fraud rate (n_fraud / n_tx)'); axes[1].set_ylabel('Count')
axes[1].axvline(PRIOR_A/(PRIOR_A+PRIOR_B), color='black', ls='--', label='Prior mean (3.5%)')
axes[1].legend(fontsize=8)

warm_pool = cs[cs.n_tx >= 10]
cold_pool = cs[(cs.n_tx >= 2) & (cs.n_tx <= 5)]
sizes = [len(warm_pool), len(cold_pool), len(cs) - len(warm_pool) - len(cold_pool)]
axes[2].pie(sizes, labels=['Warm (≥10 tx)', 'Cold (2–5 tx)', 'Excluded'], 
            colors=['#3498DB','#E74C3C','#BDC3C7'], autopct='%1.0f%%', startangle=90)
axes[2].set_title('Card Pool Split', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/nb2_data_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nWarm cards: {len(warm_pool):,} | Cold cards: {len(cold_pool):,}')

## Step 2: The Beta-Binomial Bayesian Model

Unlike the Gaussian model in Notebook 1, fraud rate estimation uses a **Beta-Binomial conjugate model**.

This is appropriate because:
- Each transaction is binary: fraud (1) or not (0)
- We're estimating a probability p ∈ [0,1]
- The Beta distribution is the natural prior for a probability

**Prior**: Beta(0.5, 13.5) — encodes our belief that a new card has ~3.5% fraud rate

**Update rule** after observing k frauds in n transactions:
```
posterior_alpha = 0.5 + k
posterior_beta  = 13.5 + (n - k)
posterior_mean  = alpha / (alpha + beta)
posterior_sigma = sqrt(alpha·beta / ((alpha+beta)²·(alpha+beta+1)))
```

As n grows, the posterior tightens around the true fraud rate.

In [ ]:
def beta_post(k_obs, n_obs):
    """Beta-Binomial posterior mean and sigma."""
    a = PRIOR_A + k_obs
    b = PRIOR_B + n_obs - k_obs
    mu    = a / (a + b)
    sigma = np.sqrt(a*b / ((a+b)**2 * (a+b+1)))
    return float(mu), float(sigma)

# Visualise posterior updating for different observation counts
x = np.linspace(0, 0.3, 300)
true_rate = 0.10  # example card with 10% fraud rate

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
scenarios = [(0,0,'Prior only'), (1,10,'1 fraud in 10 tx'), 
             (2,20,'2 fraud in 20 tx'), (5,50,'5 fraud in 50 tx')]
colors_s = ['#95A5A6','#3498DB','#2ECC71','#E74C3C']
for (k,n,label), col in zip(scenarios, colors_s):
    a = PRIOR_A + k; b = PRIOR_B + n - k
    axes[0].plot(x, beta_dist.pdf(x, a, b), color=col, lw=2, label=label)
axes[0].axvline(true_rate, color='black', ls='--', lw=1.5, label=f'True rate ({true_rate:.0%})')
axes[0].set_title('Posterior Tightening with More Data', fontweight='bold')
axes[0].set_xlabel('Fraud rate estimate'); axes[0].set_ylabel('Probability density')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

ns = list(range(0, T_STEPS+1))
mus_r, sgs_r = [], []
for n in ns:
    k = round(n * true_rate)
    m, s = beta_post(k, n)
    mus_r.append(m); sgs_r.append(s)
axes[1].plot(ns, mus_r, 'b-', lw=2, label='Posterior mean')
axes[1].fill_between(ns, [m-s for m,s in zip(mus_r,sgs_r)],
                          [m+s for m,s in zip(mus_r,sgs_r)], alpha=0.2)
axes[1].axhline(true_rate, color='red', ls='--', label=f'True rate ({true_rate:.0%})')
axes[1].axhline(PRIOR_A/(PRIOR_A+PRIOR_B), color='gray', ls=':', label='Prior mean (3.5%)')
axes[1].set_title('Estimate Convergence on Real Fraud Rates', fontweight='bold')
axes[1].set_xlabel('Transactions observed'); axes[1].set_ylabel('Fraud rate estimate')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/nb2_bayesian_update.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 3: Placement Strategies (same as Notebook 1, adapted for fraud context)

In fraud ranking: **higher score = more risky = higher in queue** (rank 0 = most suspicious).

LCB penalises uncertainty downward — a new card with uncertain fraud estimate gets placed *lower* in the fraud review queue until confidence grows.

In [ ]:
def place(wr_sorted, mu, sigma, t, strat, rng):
    """Place cold card into fraud review queue (wr_sorted: descending fraud risk)."""
    if strat == 'Naive':
        return min(int(np.searchsorted(-wr_sorted, -mu)), N_WARM)
    if strat == 'LCB':
        lcb = max(mu - LCB_K*sigma, 0.0)
        return min(int(np.searchsorted(-wr_sorted, -lcb)), N_WARM)
    if strat == 'Tiered':
        if t < GRAD_THRESH:
            lo, hi = min(TIER_BAND[0], N_WARM), min(TIER_BAND[1], N_WARM)
            return int(rng.integers(lo, hi+1))
        lcb = max(mu - LCB_K*sigma, 0.0)
        return min(int(np.searchsorted(-wr_sorted, -lcb)), N_WARM)
    return int(rng.integers(0, N_WARM+1))

print('Strategy functions defined.')
print()
print('Example: cold card with posterior mean=0.08 (8% fraud), sigma=0.04')
ws_ex = np.sort(np.random.default_rng(1).beta(0.5,13.5,N_WARM))[::-1]
rng_ex = np.random.default_rng(1)
for s in STRATEGIES:
    pos = place(ws_ex, 0.08, 0.04, 5, s, rng_ex)
    print(f'  {s:8s} → position {pos} out of {N_WARM}')

## Step 4: Run the Experiment

In [ ]:
def dcg(s,k):
    s=np.asarray(s)[:k]
    return float(np.sum(s/np.log2(np.arange(2,len(s)+2)))) if len(s) else 0.
def ndcg_k(merged,ideal,k):
    id_=dcg(np.sort(np.asarray(ideal))[::-1],k)
    return dcg(merged,k)/id_ if id_>0 else 1.
def rd_fn(pre,pos):
    sh=np.where(pre>=pos,pre+1,pre); return float(np.abs(sh-pre).mean())
def lss_fn(pre,pos):
    po=np.where(pre>=pos,pre+1,pre); tau,_=kendalltau(pre,po); return float(tau)
def ptkr_fn(pos,op,sg,thr=0.05):
    return int(pos<K and sg>thr and pos<op)

warm_pool_data = cs[cs.n_tx >= 10].reset_index(drop=True)
cold_pool_data = cs[(cs.n_tx >= 2) & (cs.n_tx <= 5)].reset_index(drop=True)

def run_once(strat, rng):
    warm = warm_pool_data.sample(N_WARM, random_state=int(rng.integers(0,99999)))
    cold = cold_pool_data.sample(N_COLD, random_state=int(rng.integers(0,99999)))
    wr   = np.sort(warm.true_rate.values)[::-1]
    pre  = np.arange(N_WARM)
    rows = []
    for t in range(T_STEPS+1):
        nv,rv,lv,pv,cv,sv=[],[],[],[],[],[]
        for _,c in cold.iterrows():
            k_obs = int(rng.binomial(t, c.true_rate)) if t>0 else 0
            mu,sg = beta_post(k_obs, t)
            pos   = place(wr, mu, sg, t, strat, rng)
            op    = int(np.searchsorted(-wr, -c.true_rate))
            merged= np.insert(wr, pos, c.true_rate)
            ideal = np.sort(np.append(wr, c.true_rate))[::-1]
            nv.append(ndcg_k(merged,ideal,K))
            rv.append(rd_fn(pre[:K],pos))
            lv.append(lss_fn(pre,pos))
            pv.append(ptkr_fn(pos,op,sg))
            cv.append(1 if abs(pos-op)<=CONV_TOL else 0)
            sv.append(sg)
        rows.append({'t':t,'strategy':strat,
                     'ndcg':np.mean(nv),'rd':np.mean(rv),'lss':np.mean(lv),
                     'ptkr':np.mean(pv),'conv':np.mean(cv),'sigma':np.mean(sv)})
    return pd.DataFrame(rows)

print(f'Running {N_RUNS} runs × {len(STRATEGIES)} strategies on real data...')
all_dfs=[]
for ri in range(N_RUNS):
    rng=np.random.default_rng(SEED+ri)
    for s in STRATEGIES:
        d=run_once(s,rng); d['run']=ri; all_dfs.append(d)
    if (ri+1)%5==0: print(f'  {ri+1}/{N_RUNS}')

full=pd.concat(all_dfs,ignore_index=True)
agg=(full.groupby(['strategy','t'])
    .agg(ndcg_m=('ndcg','mean'),ndcg_s=('ndcg','std'),
         rd_m=('rd','mean'),rd_s=('rd','std'),
         lss_m=('lss','mean'),lss_s=('lss','std'),
         ptkr_m=('ptkr','mean'),conv_m=('conv','mean'),
         sigma_m=('sigma','mean')).reset_index())
print('Done.')

## Step 5: Results

In [ ]:
rows=[]
for s in STRATEGIES:
    sub=agg[agg.strategy==s].sort_values('t')
    hit=sub[sub.conv_m>=0.80]; t80=int(hit.t.iloc[0]) if len(hit) else f'>{T_STEPS}'
    rows.append({'Strategy':s,
        'NDCG@20 t=0':round(sub[sub.t==0].ndcg_m.iloc[0],4),
        'NDCG@20 t=20':round(sub[sub.t==T_STEPS].ndcg_m.iloc[0],4),
        'RD t=0':round(sub[sub.t==0].rd_m.iloc[0],4),
        'RD t=20':round(sub[sub.t==T_STEPS].rd_m.iloc[0],4),
        'PTKR peak':round(sub.ptkr_m.max(),4),
        '80% Conv':t80})
tbl=pd.DataFrame(rows)
print('=== IEEE-CIS Summary ===')
print(tbl.to_string(index=False))

In [ ]:
def shade(ax,strat,ycol,scol):
    sub=agg[agg.strategy==strat].sort_values('t')
    t=sub.t.values; mu=sub[ycol].values; sd=sub[scol].values
    c=COLORS[strat]; ax.plot(t,mu,color=c,lw=2.2,label=strat)
    ax.fill_between(t,mu-sd,mu+sd,color=c,alpha=0.13)

fig=plt.figure(figsize=(18,12))
fig.suptitle('IEEE-CIS Real Data: Cold-Start Placement Strategy Comparison\n'
    f'N_warm={N_WARM}, N_cold={N_COLD} real cards, K={K}, {N_RUNS} MC runs',
    fontsize=13,fontweight='bold',y=0.98)
gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.45,wspace=0.35)

for spec,ycol,scol,title,ylabel in [
    (gs[0,0],'ndcg_m','ndcg_s','NDCG@20','NDCG@20 (↑ better)'),
    (gs[0,1],'rd_m','rd_s','Rank Displacement','Mean |rank shift| top-20 warm cards (↓ better)'),
    (gs[0,2],'lss_m','lss_s','List Stability (Kendall τ)','Kendall τ (↑ more stable)')]:
    ax=fig.add_subplot(spec)
    for s in STRATEGIES: shade(ax,s,ycol,scol)
    ax.set_title(title,fontsize=11,fontweight='bold')
    ax.set_xlabel('Transactions revealed (t)'); ax.set_ylabel(ylabel,fontsize=9)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax4=fig.add_subplot(gs[1,0])
for s in STRATEGIES:
    sub=agg[agg.strategy==s].sort_values('t')
    ax4.plot(sub.t,sub.ptkr_m,color=COLORS[s],lw=2.2,label=s)
ax4.set_title('Premature Top-K Rate (PTKR)',fontsize=11,fontweight='bold')
ax4.set_xlabel('Transactions revealed (t)'); ax4.set_ylabel('Fraction (↓ better)',fontsize=9)
ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

ax5=fig.add_subplot(gs[1,1])
for s in STRATEGIES:
    sub=agg[agg.strategy==s].sort_values('t')
    ax5.plot(sub.t,sub.conv_m,color=COLORS[s],lw=2.2,label=s)
ax5.axhline(0.80,color='black',ls='--',lw=1.2,label='80% threshold')
ax5.set_title('Convergence Rate',fontsize=11,fontweight='bold')
ax5.set_xlabel('Transactions revealed (t)')
ax5.set_ylabel(f'Fraction within ±{CONV_TOL} ranks of oracle',fontsize=9)
ax5.set_ylim(0,1.05); ax5.legend(fontsize=9); ax5.grid(alpha=0.3)

ax6=fig.add_subplot(gs[1,2])
sub=agg[agg.strategy=='LCB'].sort_values('t')
ax6.plot(sub.t,sub.sigma_m,color='#8E44AD',lw=2.2)
ax6.fill_between(sub.t,0,sub.sigma_m,color='#8E44AD',alpha=0.15)
ax6.axhline(0.05,color='black',ls='--',lw=1.2,label='σ threshold (0.05)')
ax6.set_title('Posterior σ Decay (Real Data)',fontsize=11,fontweight='bold')
ax6.set_xlabel('Transactions revealed (t)'); ax6.set_ylabel('Mean posterior σ')
ax6.legend(fontsize=9); ax6.grid(alpha=0.3)

plt.savefig('figures/nb2_all_metrics.png',dpi=150,bbox_inches='tight')
plt.show()

## Step 6: Synthetic vs Real — Side by Side

| Metric | Synthetic (NB1) | IEEE-CIS (NB2) | Consistent? |
|---|---|---|---|
| Naive PTKR peak | 28.2% | 0.17% | ✓ (lower due to low base fraud rate) |
| LCB/Tiered PTKR | 0% | **0%** | ✓ |
| RD: Tiered vs Naive | −55% | −19% | ✓ |
| NDCG cost | ~0 | ~0 | ✓ |
| Tiered convergence | t=20+ | t=8 | Both at graduation threshold |

The real-data PTKR being lower than synthetic is **expected** — real fraud rates are only 3.5%, so most new cards genuinely belong low in the queue. The synthetic experiment deliberately tested high-risk cold entities to stress-test the effect.

Both datasets confirm the same story: **conservative placement costs nothing in list quality and meaningfully reduces disruption.**
